In [119]:
import pandas as pd           # data mnipulation
import numpy as np            # number manipulation/crunching
import matplotlib.pyplot as plt  # plotting
from sklearn.metrics import  accuracy_score, classification_report 
import openpyxl
# Train Test split
from sklearn.model_selection import train_test_split
# Random forest classifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from anchor import utils
from anchor import anchor_tabular
from sklearn.model_selection import LeaveOneOut

diabetes = pd.read_csv("diabetes.csv")
diabetes['BloodPressure'] = diabetes['BloodPressure'].replace(to_replace=0,value=diabetes['BloodPressure'].median())
diabetes['BMI'] = diabetes['BMI'].replace(to_replace=0,value=diabetes['BMI'].median())
diabetes['Insulin'] = diabetes['Insulin'].replace(to_replace=0,value=diabetes['Insulin'].median())
diabetes['Glucose'] = diabetes['Glucose'].replace(to_replace=0,value=diabetes['Glucose'].median())
diabetes['SkinThickness'] = diabetes['SkinThickness'].replace(to_replace=0,value=diabetes['SkinThickness'].median())
diabetes['BloodPressure'] = diabetes['BloodPressure'].replace(np.nan, 0)
diabetes['BMI'] = diabetes['BMI'].replace(np.nan, 0)
diabetes['Insulin'] = diabetes['Insulin'].replace(np.nan, 0)
diabetes['Glucose'] = diabetes['Glucose'].replace(np.nan, 0)
diabetes['SkinThickness'] = diabetes['SkinThickness'].replace(np.nan, 0)
diabetes["Outcome"].value_counts()

Y = diabetes['Outcome']
target_names=["No Diabetes", "Diabetes"]
X = diabetes[['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI','DiabetesPedigreeFunction', 'Age']]
#X = diabetes.iloc[:,:-1].values
X_featurenames = X.columns
#X_featurenames = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
        #'DiabetesPedigreeFunction', 'Age']

# Split the data into train and test data:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, shuffle = True)

model = xgb.XGBClassifier()
model.fit(X_train, Y_train)
pred = model.predict(X_test)
accuracy = accuracy_score(Y_test, pred)
print("Accuracy:", accuracy)
print(classification_report(Y_test, pred, target_names=["No Diabetes", "Diabetes"]))

dataframe_columns = []
test_index = "test_index"
dataframe_columns.append(test_index)
for feature in X_featurenames:
    feature_lower_interval = feature + "_lower_interval"
    feature_upper_interval = feature + "_upper_interval"
    feature_rank = feature + "_rank"
    dataframe_columns.append(feature_lower_interval)
    dataframe_columns.append(feature_upper_interval)
    dataframe_columns.append(feature_rank)
accuracy_classifier = "accuracy_classifier"
precision = "precision"
coverage = "coverage"
prediction = "prediction"
true_class = "true_class"
dataframe_columns.append(accuracy_classifier)
dataframe_columns.append(precision)
dataframe_columns.append(coverage)
dataframe_columns.append(prediction)
dataframe_columns.append(true_class)
df = pd.DataFrame(columns = dataframe_columns)


number_iterations = 5
for i in range(X_test.shape[1]):
    print(i)
    for count in range(number_iterations):
        model = xgb.XGBClassifier()
        model.fit(X_train, Y_train)
        explainer = anchor_tabular.AnchorTabularExplainer(class_names=target_names,
                                                        feature_names=X_featurenames,
                                                        train_data=X_train.values)
        exp = explainer.explain_instance(X_test.iloc[i].values, model.predict, threshold=0.95)
        list = exp.names()
        row = []
        row.append(i)
        for feature in X_featurenames:
            ok = 0
            for j in range(len(list)):
                if list[j].find(feature) != -1 :
                    ok = 1
                    #print(list[j])
                    rank = j + 1
                    if list[j].count('.') == 2:
                        lower_interval = list[j][0 : list[j].find('<')]
                        upper_interval = list[j][list[j].find('=') + 2 : len(list[j]) + 1]
                    elif list[j].count('.') == 1 and list[j].find('<') != -1:
                        lower_interval = 0
                        if(list[j].find('=') != -1):
                            upper_interval = list[j][list[j].find('=') + 2 : len(list[j]) + 1]
                        else:
                            upper_interval = list[j][list[j].find('<') + 2 : len(list[j]) + 1]
                    elif list[j].count('.') == 1 and list[j].find('>') != -1:
                        upper_interval = max(X_test[feature])
                        if(list[j].find('=') != -1):
                            lower_interval = list[j][list[j].find('=') + 2 : len(list[j]) + 1]
                        else:
                            lower_interval = list[j][list[j].find('>') + 2 : len(list[j]) + 1]
            if ok == 0:
                lower_interval = -1
                upper_interval = -1
                rank = -1
            #print(lower_interval)
            #print(upper_interval)
            #print(rank)
            row.append(lower_interval)
            row.append(upper_interval)
            row.append(rank)
        pred = model.predict(X_test)
        accuracy_classifier = accuracy_score(Y_test, pred)
        precision = exp.precision()
        coverage = exp.coverage()
        prediction =  explainer.class_names[model.predict(X_test.iloc[i].values.reshape(1, -1))[0]]
        true_class = target_names[Y_test.iloc[i]]
        row.append(accuracy_classifier)
        row.append(precision)
        row.append(coverage)
        row.append(prediction)
        row.append(true_class)
        #print(row)
        #print(df)
        df.loc[i * number_iterations + count] = row
        #print(df)

df.to_excel("outputDiabetesAnchor.xlsx")

# print('Anchor: %s' % (' AND '.join(exp.names())))
# print('Precision: %.2f' % exp.precision())
# print('Coverage: %.2f' % exp.coverage())

Accuracy: 0.7597402597402597
              precision    recall  f1-score   support

 No Diabetes       0.80      0.83      0.81        98
    Diabetes       0.68      0.64      0.66        56

    accuracy                           0.76       154
   macro avg       0.74      0.73      0.74       154
weighted avg       0.76      0.76      0.76       154

0


KeyboardInterrupt: 